# Lelapa AI Buzuzu-Mavi Challenge

**Hausa and Swahili sentiment analysis, AfriXNLI, and English-to-Hausa/Swahili translation** using `lelapa/InkubaLM-0.4B`. Run cells in order after supplying the challenge Parquet files. All checkpoints and predictions are written beneath `OUTPUT_DIR`.


## 1. Environment and configuration

In [ ]:
# Run only when dependencies are absent, then restart the notebook kernel.
# %pip install "transformers==4.45.2" "peft==0.13.2" "datasets==3.0.2" "accelerate==1.0.1" "pyarrow" "sacrebleu"

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_NAME = "lelapa/InkubaLM-0.4B"
DATA_DIR = Path(os.environ.get("LELAPA_DATA_DIR", "./data")).expanduser()
OUTPUT_DIR = Path(os.environ.get("LELAPA_OUTPUT_DIR", "./output")).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Keep False for static review. Set True only after reviewing the model repository code.
TRUST_REMOTE_CODE = False
HF_TOKEN = os.environ.get("HF_TOKEN") or None

DATA_FILES = {
    "sentiment": ("Sentiment Analysis/Sentiment Analysis_train.parquet", "Sentiment Analysis/Sentiment Analysis_test.parquet"),
    "xnli": ("AfriXNLI/AfriXNLI_train.parquet", "AfriXNLI/AfriXNLI_test.parquet"),
    "mt": ("Machine translation/MT_train.parquet", "Machine translation/MT_test.parquet"),
}

def load_split(task, split):
    index = {"train": 0, "test": 1}[split]
    path = DATA_DIR / DATA_FILES[task][index]
    if not path.is_file():
        raise FileNotFoundError(f"Missing {task} {split} data: {path}")
    return pd.read_parquet(path)

## 2. Task definitions and label conventions

**sentiment: 0 positive, 1 neutral, 2 negative** and **AfriXNLI: 0 neither, 1 true, 2 false**. 

In [ ]:
TASK_INTRO = (
    "Below is an instruction that describes a task, paired with an input that "
    "provides further context. Write a response that appropriately completes the request."
)
XNLI_LABELS = {0: "neither", 1: "true", 2: "false"}
SENTIMENT_LABELS = {
    "hausa": {0: "kyakkyawa", 1: "tsaka-tsaki", 2: "korau"},
    "swahili": {0: "chanya", 1: "wastani", 2: "hasi"},
}
SENTIMENT_ALIASES = {
    "positive": 0, "neutral": 1, "negative": 2,
    "kyakkyawa": 0, "tsaka-tsaki": 1, "korau": 2,
    "chanya": 0, "wastani": 1, "hasi": 2,
}

def sentiment_target(row):
    value = row["targets"]
    if pd.isna(value):
        raise ValueError("Sentiment training row is missing a target")
    if isinstance(value, (int, np.integer)) or (isinstance(value, float) and value.is_integer()):
        label_id = int(value)
    else:
        label_id = SENTIMENT_ALIASES[str(value).strip().casefold()]
    return SENTIMENT_LABELS[str(row["langs"]).casefold()][label_id]

def xnli_target(row):
    value = row["targets"]
    if pd.isna(value):
        raise ValueError("XNLI training row is missing a target")
    return XNLI_LABELS[int(value)]

def build_prompt(task, row):
    instruction = str(row.get("instruction", "")).strip()
    if task == "sentiment":
        return f"{TASK_INTRO}\n\n### Instruction: {instruction}\n\n### Text: {row['inputs']}\n\n### Response:"
    if task == "xnli":
        instruction = instruction or "Given the premise, is the following question True, False or Neither?"
        return (f"{TASK_INTRO}\n\n### Instruction: {instruction}\n\n"
                f"### Premise: {row['premise']}\n\n### Question: {row['inputs']}\n\n### Response:")
    if task == "mt":
        instruction = instruction or f"Translate the following from English into {'Hausa' if row['langs'] == 'eng-hau' else 'Swahili'}."
        return (f"{TASK_INTRO}\n\n### Instruction: {instruction} "
                f"Do not produce any extra text besides the translation.\n\n"
                f"### Text: {row['inputs']}\n\n### Translated Text:")
    raise ValueError(f"Unknown task: {task}")

def target_text(task, row):
    if task == "sentiment":
        return sentiment_target(row)
    if task == "xnli":
        return xnli_target(row)
    if pd.isna(row["targets"]):
        raise ValueError("Translation training row is missing a target")
    return str(row["targets"])

## 3. Load model and prepare supervised examples

Each task starts with its **own fresh base model** so that adapters have independent weights. The response-only mask ignores the prompt; it is derived from the tokenized prompt length rather than a character offset. The collator pads labels with `-100`, the ignored value for causal language modeling. Long examples keep the response and trim the earliest prompt context.

In [ ]:
def make_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, token=HF_TOKEN, trust_remote_code=TRUST_REMOTE_CODE
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer

def make_base_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=HF_TOKEN, trust_remote_code=TRUST_REMOTE_CODE
    )
    model.config.use_cache = False  # Required for gradient checkpointing, if enabled later.
    return model

def discover_lora_targets(model):
    # Attention projections vary among model implementations; select actual module names.
    module_names = {name.split(".")[-1] for name, _ in model.named_modules()}
    for pair in (("q_proj", "v_proj"), ("query_key_value",), ("c_attn",)):
        if all(name in module_names for name in pair):
            return list(pair)
    raise ValueError("No known attention projection modules found; inspect model.named_modules() before configuring LoRA")

def make_adapter():
    base = make_base_model()
    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=256,
        lora_alpha=512,
        lora_dropout=0.05,
        target_modules=discover_lora_targets(base),
    )
    return get_peft_model(base, config)

def tokenize_supervised(task, frame, tokenizer, max_length=512):
    records = []
    for row in frame.to_dict("records"):
        prompt = build_prompt(task, row) + " "
        response = target_text(task, row)
        prefix = tokenizer(prompt, add_special_tokens=True)["input_ids"]
        continuation = tokenizer(response, add_special_tokens=False)["input_ids"]
        if tokenizer.eos_token_id is not None:
            continuation.append(tokenizer.eos_token_id)
        if not continuation:
            raise ValueError(f"Empty tokenized response for ID={row.get('ID')}")
        if len(continuation) >= max_length:
            raise ValueError("Response exceeds max_length; increase the context limit")
        prefix = prefix[-(max_length - len(continuation)):]
        ids = prefix + continuation
        records.append({"input_ids": ids, "attention_mask": [1] * len(ids),
                        "labels": [-100] * len(prefix) + continuation})
    return Dataset.from_list(records)

class CausalResponseCollator:
    def __init__(self, tokenizer):
        self.pad_id = tokenizer.pad_token_id

    def __call__(self, examples):
        size = max(len(item["input_ids"]) for item in examples)
        return {key: torch.tensor([
            item[key] + [fill] * (size - len(item[key])) for item in examples
        ], dtype=torch.long) for key, fill in (
            ("input_ids", self.pad_id), ("attention_mask", 0), ("labels", -100)
        )}

## 4. Train and save the three task adapters

Both languages for each task are pooled before training. This prevents a second `Trainer` from continuing the first language's adapter while using a new optimizer schedule without documenting it. The original notebook used 30 sentiment epochs at `5e-5`, 10 XNLI epochs at `5e-4`, and 5 translation epochs at `5e-5`; these are preserved as editable experiment settings. Training requires the three local training files and enough GPU memory for a 0.4B parameter model plus LoRA rank 256.

In [ ]:
TRAIN_SETTINGS = {
    "sentiment": {"epochs": 30, "lr": 5e-5, "weight_decay": 0.001},
    "xnli": {"epochs": 10, "lr": 5e-4, "weight_decay": 0.01},
    "mt": {"epochs": 5, "lr": 5e-5, "weight_decay": 0.01},
}
ADAPTER_DIRS = {task: OUTPUT_DIR / f"{task}_adapter" for task in TRAIN_SETTINGS}

def train_task(task, tokenizer):
    if task not in TRAIN_SETTINGS:
        raise ValueError(task)
    frame = load_split(task, "train")
    dataset = tokenize_supervised(task, frame, tokenizer)
    model = make_adapter()
    settings = TRAIN_SETTINGS[task]
    args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / f"{task}_checkpoints"),
        per_device_train_batch_size=1,
        num_train_epochs=settings["epochs"],
        learning_rate=settings["lr"],
        weight_decay=settings["weight_decay"],
        save_total_limit=2,
        eval_strategy="no",
        logging_steps=25,
        report_to="none",
        remove_unused_columns=False,
        seed=SEED,
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset,
                      data_collator=CausalResponseCollator(tokenizer))
    trainer.train()
    model.save_pretrained(str(ADAPTER_DIRS[task]))
    tokenizer.save_pretrained(str(ADAPTER_DIRS[task]))
    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tokenizer = make_tokenizer()
# Uncomment to train. Training is intentionally not triggered by opening the notebook.
# for task_name in ("sentiment", "xnli", "mt"):
#     train_task(task_name, tokenizer)

## 5. Restore the optional sequence classifier

The final cells of the source notebook loaded a three-class `AutoModelForSequenceClassification` checkpoint but mixed its logits with causal LM token scores and referred to an undefined `Upd_model`. This independent sentiment baseline uses class IDs **0 positive, 1 neutral, 2 negative** and trains and tests with its own tokenizer and checkpoint. It is an alternative to the sentiment LoRA adapter.

In [ ]:
CLASSIFIER_DIR = OUTPUT_DIR / "sentiment_classifier"

def build_classifier_dataset(frame, classifier_tokenizer, max_length=512, labeled=True):
    prompts = [build_prompt("sentiment", row) for row in frame.to_dict("records")]
    encoded = classifier_tokenizer(prompts, truncation=True, max_length=max_length)
    if labeled:
        for row, label in zip(frame.to_dict("records"),
                              [sentiment_target(row) for row in frame.to_dict("records")]):
            encoded.setdefault("labels", []).append(SENTIMENT_ALIASES[label])
    return Dataset.from_dict(encoded)

def train_sentiment_classifier():
    clf_tokenizer = make_tokenizer()
    clf_tokenizer.padding_side = "right"
    classifier = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, token=HF_TOKEN, trust_remote_code=TRUST_REMOTE_CODE,
        id2label={0: "positive", 1: "neutral", 2: "negative"},
        label2id={"positive": 0, "neutral": 1, "negative": 2},
    )
    classifier.config.pad_token_id = clf_tokenizer.pad_token_id
    dataset = build_classifier_dataset(load_split("sentiment", "train"), clf_tokenizer)
    trainer = Trainer(
        model=classifier,
        args=TrainingArguments(
            output_dir=str(OUTPUT_DIR / "sentiment_classifier_checkpoints"),
            num_train_epochs=5, learning_rate=2e-5, per_device_train_batch_size=1,
            save_total_limit=2, report_to="none", seed=SEED,
        ),
        train_dataset=dataset,
        data_collator=DataCollatorWithPadding(clf_tokenizer),
    )
    trainer.train()
    classifier.save_pretrained(str(CLASSIFIER_DIR))
    clf_tokenizer.save_pretrained(str(CLASSIFIER_DIR))

def predict_sentiment_classifier(frame):
    clf_tokenizer = AutoTokenizer.from_pretrained(str(CLASSIFIER_DIR))
    classifier = AutoModelForSequenceClassification.from_pretrained(str(CLASSIFIER_DIR)).to(DEVICE).eval()
    records = []
    for row in frame.to_dict("records"):
        inputs = clf_tokenizer(build_prompt("sentiment", row), return_tensors="pt",
                               truncation=True, max_length=512).to(DEVICE)
        with torch.inference_mode():
            label_id = int(classifier(**inputs).logits.argmax(dim=-1).item())
        records.append({"ID": row["ID"], "Response": label_id})
    return pd.DataFrame(records)

# train_sentiment_classifier()

## 6. Inference across all three adapters


In [ ]:
def load_task_adapter(task):
    checkpoint = ADAPTER_DIRS[task]
    if not (checkpoint / "adapter_config.json").is_file():
        raise FileNotFoundError(f"Train the {task} adapter first: {checkpoint}")
    adapter = PeftModel.from_pretrained(make_base_model(), str(checkpoint))
    return adapter.to(DEVICE).eval()

def response_log_likelihood(model, tokenizer, prompt, response):
    prefix = tokenizer(prompt + " ", add_special_tokens=True)["input_ids"]
    continuation = tokenizer(response, add_special_tokens=False)["input_ids"]
    if not prefix or not continuation:
        raise ValueError("Prompt or candidate response produced no tokens")
    ids = torch.tensor([prefix + continuation], device=DEVICE)
    with torch.inference_mode():
        logits = model(input_ids=ids, attention_mask=torch.ones_like(ids)).logits
        positions = torch.arange(len(prefix) - 1, len(prefix) + len(continuation) - 1, device=DEVICE)
        log_probs = F.log_softmax(logits[0, positions, :].float(), dim=-1)
        candidate_ids = torch.tensor(continuation, device=DEVICE)
        return float(log_probs.gather(1, candidate_ids[:, None]).sum().item())

def predict_category(model, tokenizer, task, row):
    prompt = build_prompt(task, row)
    options = SENTIMENT_LABELS[str(row["langs"]).casefold()] if task == "sentiment" else XNLI_LABELS
    scores = {label_id: response_log_likelihood(model, tokenizer, prompt, text)
              for label_id, text in options.items()}
    return max(scores, key=scores.get)

def predict_translation(model, tokenizer, row, max_new_tokens=100):
    inputs = tokenizer(build_prompt("mt", row) + " ", return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        output = model.generate(
            **inputs, do_sample=False, max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def predict_task(task, frame, tokenizer):
    model = load_task_adapter(task)
    try:
        rows = []
        for row in frame.to_dict("records"):
            response = (predict_translation(model, tokenizer, row) if task == "mt"
                        else predict_category(model, tokenizer, task, row))
            record = {"ID": row["ID"], "Response": response}
            if "targets" in row and pd.notna(row["targets"]):
                record["Targets"] = row["targets"]
            rows.append(record)
        return pd.DataFrame(rows)
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def make_submission(tokenizer, split="test", sentiment_backend="adapter"):
    if split not in {"test", "train"}:
        raise ValueError("split must be 'test' or 'train'")
    if sentiment_backend not in {"adapter", "classifier"}:
        raise ValueError("sentiment_backend must be 'adapter' or 'classifier'")
    parts = []
    for task in ("sentiment", "xnli", "mt"):
        frame = load_split(task, split)
        predicted = (predict_sentiment_classifier(frame) if task == "sentiment" and sentiment_backend == "classifier"
                     else predict_task(task, frame, tokenizer))
        if len(predicted) != len(frame) or not predicted["ID"].equals(frame["ID"].reset_index(drop=True)):
            raise ValueError(f"Prediction rows do not match the {task} data")
        parts.append(predicted[["ID", "Response"]])
    result = pd.concat(parts, ignore_index=True)
    if result["ID"].duplicated().any() or result["Response"].isna().any():
        raise ValueError("Submission has duplicate IDs or missing predictions")
    destination = OUTPUT_DIR / ("submission.csv" if split == "test" else "submission_train_check.csv")
    result.to_csv(destination, index=False)
    return result

# submission = make_submission(tokenizer, split="test")
# submission.head()

## 7. Optional evaluation on held-out labeled data

In [ ]:
def evaluate_labeled(task, labeled_frame, tokenizer):
    if labeled_frame.empty or labeled_frame["targets"].isna().any():
        raise ValueError("Evaluation requires nonempty data with every target present")
    predictions = predict_task(task, labeled_frame, tokenizer)
    if task == "sentiment":
        truth = [SENTIMENT_ALIASES[sentiment_target(row)] for row in labeled_frame.to_dict("records")]
        return {"accuracy": float(np.mean(predictions["Response"].to_numpy() == np.array(truth)))}
    if task == "xnli":
        truth = labeled_frame["targets"].astype(int).to_numpy()
        return {"accuracy": float(np.mean(predictions["Response"].to_numpy() == truth))}
    import sacrebleu
    return {"bleu": float(sacrebleu.corpus_bleu(
        predictions["Response"].astype(str).tolist(),
        [labeled_frame["targets"].astype(str).tolist()],
    ).score)}

# dev = pd.read_parquet(DATA_DIR / "path/to/labeled_dev.parquet")
# evaluate_labeled("xnli", dev, tokenizer)

## 8. Experimental BLEU reward training (optional)

The source notes attempted to backpropagate through `generate()` and read `.logits` from generated token IDs. That is not a valid training loss. 

In [ ]:
def reinforce_mt_step(model, tokenizer, row, optimizer, max_new_tokens=48):
    import sacrebleu

    prompt_ids = tokenizer(build_prompt("mt", row) + " ", return_tensors="pt")["input_ids"].to(DEVICE)
    reference = str(row["targets"])
    model.eval()  # Disable dropout when evaluating sampled sequences and their log probabilities.
    with torch.no_grad():
        sampled = model.generate(input_ids=prompt_ids, do_sample=True, top_p=0.9,
                                 temperature=1.0, max_new_tokens=max_new_tokens,
                                 pad_token_id=tokenizer.pad_token_id)
        greedy = model.generate(input_ids=prompt_ids, do_sample=False,
                                max_new_tokens=max_new_tokens,
                                pad_token_id=tokenizer.pad_token_id)
    prompt_len = prompt_ids.shape[1]
    sampled_ids = sampled[:, prompt_len:]
    sampled_text = tokenizer.decode(sampled_ids[0], skip_special_tokens=True).strip()
    greedy_text = tokenizer.decode(greedy[0, prompt_len:], skip_special_tokens=True).strip()
    reward = sacrebleu.sentence_bleu(sampled_text, [reference]).score / 100.0
    baseline = sacrebleu.sentence_bleu(greedy_text, [reference]).score / 100.0
    if sampled_ids.shape[1] == 0:
        raise ValueError("Sampling returned no translation tokens")

    model.train()
    # logits at prefix_length - 1 predict the first sampled token.
    logits = model(input_ids=sampled, attention_mask=torch.ones_like(sampled)).logits
    token_logits = logits[:, prompt_len - 1:-1, :].float()
    log_probs = F.log_softmax(token_logits, dim=-1).gather(
        -1, sampled_ids.unsqueeze(-1)
    ).squeeze(-1)
    if tokenizer.eos_token_id is not None:
        eos = sampled_ids.eq(tokenizer.eos_token_id)
        valid = (eos.cumsum(dim=1) - eos == 0)
        log_probs = log_probs * valid
    loss = -(reward - baseline) * log_probs.sum()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return {"loss": float(loss.item()), "sample_bleu": reward * 100,
            "greedy_bleu": baseline * 100}

def train_mt_with_bleu_reward(tokenizer, epochs=1, limit=None, lr=1e-5):
    model = load_task_adapter("mt")
    model.config.use_cache = False
    model.train()
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=lr)
    rows = load_split("mt", "train").to_dict("records")
    if limit is not None:
        rows = rows[:limit]
    history = []
    for epoch in range(epochs):
        random.Random(SEED + epoch).shuffle(rows)
        for row in rows:
            history.append(reinforce_mt_step(model, tokenizer, row, optimizer))
    reward_dir = OUTPUT_DIR / "mt_bleu_reward_adapter"
    model.save_pretrained(str(reward_dir))
    tokenizer.save_pretrained(str(reward_dir))
    return pd.DataFrame(history)

# reward_history = train_mt_with_bleu_reward(tokenizer, epochs=1, limit=20)